In [1]:
import os,sqlite3,pandas as pd

In [2]:
def getClimateIndices(directory):
    temp = []  # temporary list/dict used throught the program 
 #   features = pd.DataFrame()  # this is a Pandas Dataframe that will hold the Climate Indices
    features={}
    #  add year-month pairs to the features dataframe that will match the available data
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file():
             #   print(entry.name)
    #  the Climate Indices are stored in flat files, so read them all in and store them in features
                file=entry.name
                file = file.rstrip("\n")
                spl = file.split(".")
                name = spl[0]
                temp = []
                tempdf = pd.DataFrame()
                print(file)
                if file.find("Zone.Identifier") == -1:
                    spl=file.split(".")
                    name=".".join(spl[:-1])
                    with open(f"{directory}/{file}","r") as fin:
                        line=fin.readline()
                        spl = line.split()
                        start = int(spl[0])
                        end = int(spl[1])
                        nl=0
                        print(start,end)
                        process=True
                        for nn in range(start,end+1):
                           line = fin.readline()
                           spl = line.split()
                           year = int(spl[0])
                           if year > 1949  and year < 2024:
                               for mon in range(1,13):
                                   val=float(spl[mon])
                                   if val < -30:
                                       process=False
                                       print(file,year,mon,val)
                        fin.close()
                    if process: 
                        with open(f"{directory}/{file}","r") as fin:
                            line=fin.readline()
                            spl = line.split()
                            start = int(spl[0])
                            end = int(spl[1])
                            nl=0
                            print(start,end)
                            process=True
                            for nn in range(start,end+1):
                               line = fin.readline()
                               spl = line.split()
                               year = int(spl[0])
                               if year not in features:
                                   features[year]={}
                                   
                               for mon in range(1,13):
                                   if mon not in features[year]:
                                       features[year][mon]={}
                                   val=float(spl[mon])
                                   features[year][mon][name]=val
                    else:
                        print("SKIPPING ",file)
                            
        return features

In [8]:
climate=getClimateIndices("/home/joe/work/Fire/ML/New/Data/Climate-Indices")


ea.data.txt
1948 2025
1948 2025
aao.data.txt
1979 2024
1979 2024
pna.data.txt
1948 2025
1948 2025
nino34.long.anom.data.txt
1870 2024
1870 2024
nao.long.data.txt
1821 2024
1821 2024
heatcentra.data.txt
1979 2025
1979 2025
nino12.long.anom.data.txt
1870 2024
1870 2024


In [13]:
data={}
data['year']=[]
data['indices']= {}
indices=[]
for yr,dct in climate.items():
    for mon,dct2 in dct.items():
        for k in dct2.keys():
            if k not in data['indices']:
                data['indices'][k]=[]
data['indices'] 

{'ea.data': [],
 'pna.data': [],
 'nino34.long.anom.data': [],
 'nao.long.data': [],
 'nino12.long.anom.data': [],
 'aao.data': [],
 'heatcentra.data': []}

In [ ]:

for yr,dct in climate.items():
    for mon,dct2 in dct.items():
       
        data['year'].append(yr*100+mon)
        for k in data['indices'].keys():
            if k in dct2:
                data['indices'][k].append(dct2[k])
            else:
                data['indices'][k].append(None)

1948 1 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.05, 'nao.long.data': 1.53, 'nino12.long.anom.data': -0.26}
1948 2 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.37, 'nao.long.data': 0.66, 'nino12.long.anom.data': 0.42}
1948 3 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.63, 'nao.long.data': 3.48, 'nino12.long.anom.data': 0.57}
1948 4 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.25, 'nao.long.data': -0.48, 'nino12.long.anom.data': 0.1}
1948 5 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.31, 'nao.long.data': -1.23, 'nino12.long.anom.data': 0.17}
1948 6 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': 0.06, 'nao.long.data': -0.27, 'nino12.long.anom.data': -0.41}
1948 7 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': -0.06, 'nao.long.data': 0.59, 'nino12.long.anom.data': -0.71}
1948 8 {'ea.data': -99.9, 'pna.data': -99.9, 'nino34.long.anom.data': -0.02, 'nao.lo

In [28]:
final=pd.DataFrame({"yrmo": data['year']} )
for k in data['indices'].keys():
    final[k]=data['indices'][k]

final = final.loc[(final['yrmo'] >= 199001) & (final['yrmo'] <= 202412) ]

In [31]:
for col in final.columns:
    tmp = final.loc[(final['yrmo'] >= 199001) & (final[col] == final[col].min()) ]
    if tmp[col].min() < -99:
        print(col,tmp['yrmo'].values[0],tmp[col].values[0])


nao.long.data 202402 -99.99


In [33]:
final.isna().sum()

yrmo                     0
ea.data                  0
pna.data                 0
nino34.long.anom.data    0
nino12.long.anom.data    0
aao.data                 0
heatcentra.data          0
dtype: int64

In [32]:
final.drop(columns=['nao.long.data'],inplace=True)

In [34]:
conn = sqlite3.connect('/home/joe/work/Fire/ML/New/Data/climate-indices.db')

# Write dataframe to SQLite table
final.to_sql('indices', conn, if_exists='replace', index=False)

# Close connection
conn.close()